In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# Sets the path to the parent directory of RR classes
HERE = Path(".")

# Import data from the O*NET database, at ISCO-08 occupation level.
# The original data uses a version of SOC classification, but the data we load here
# are already cross-walked to ISCO-08 using: https://ibs.org.pl/en/resources/occupation-classifications-crosswalks-from-onet-soc-to-isco/

# The O*NET database contains information for occupations in the USA, including
# the tasks and activities typically associated with a specific occupation.

task_data = pd.read_csv(HERE / "Data" / "onet_tasks.csv")
# isco08 variable is for occupation codes
# the t_* variables are specific tasks conducted on the job

# read employment data from Eurostat
# These datasets include quarterly information on the number of workers in specific
# 1-digit ISCO occupation categories. (Check here for details: https://www.ilo.org/public/english/bureau/stat/isco/isco08/)

isco_list = []

for i in range(1, 10):
  sheet = 'ISCO' + str(i)

  df = pd.read_excel(HERE / "Data0" / "Eurostat_employment_isco.xlsx", sheet_name = sheet)
  isco_list.append(df)

# We will focus on three countries, but perhaps we could clean this code to allow it
# to easily run for all the countries in the sample?

def get_total_workers(country_name, list_of_datasets):
  total = 0
  for df in list_of_datasets:
    total = total + df[country_name]
  return total

In [ ]:
all_data = pd.concat(isco_list, ignore_index = True)

# We have 9 occupations and the same time range for each, so we can add the totals by
# adding a vector that is 9 times the previously calculated totals
countries = ["Belgium", "Spain", "Poland"]

for country in countries:
  total_country = get_total_workers(country, isco_list)
  all_data[f"total_{country}"] = pd.concat([total_country] * 9, ignore_index = True)
  all_data[f"share_{country}"] = all_data[country] / all_data[f"total_{country}"]

In [ ]:
# Now let's look at the task data. We want the first digit of the ISCO variable only
task_data["isco08_1dig"] = task_data["isco08"].astype(str).str[:1].astype(int)

# And we'll calculate the mean task values at a 1-digit level
# (more on what these tasks are below)
aggdata = task_data.groupby(["isco08_1dig"]).mean()
aggdata = aggdata.drop(columns=["isco08"])

# We'll be interested in tracking the intensity of Non-routine cognitive analytical tasks
# Using a framework reminiscent of the work by David Autor.

#These are the ones we're interested in:
# Non-routine cognitive analytical
# 4.A.2.a.4 Analyzing Data or Information
# 4.A.2.b.2 Thinking Creatively
# 4.A.4.a.1 Interpreting the Meaning of Information for Others

#Let's combine the data.
combined = pd.merge(all_data, aggdata, left_on='ISCO', right_on='isco08_1dig', how='left')
# Traditionally, the first step is to standardise the task values using weights
# defined by share of occupations in the labour force. This should be done separately
# for each country. Standardisation -> getting the mean to 0 and std. dev. to 1.
# Let's do this for each of the variables that interests us:



In [ ]:
tasks = ["t_4A2a4", "t_4A2b2", "t_4A4a1"]

for task in tasks:
  for country in countries:

    weight_col = f"share_{country}"

    temp_mean = np.average(combined[task], weights=combined[weight_col])
    temp_sd = np.sqrt(np.average((combined[task] - temp_mean)**2, weights=combined[weight_col]))

    new_col_name = f"std_{country}_{task}"
    combined[new_col_name] = (combined[task] - temp_mean) / temp_sd


In [ ]:
# The next step is to calculate the `classic` task content intensity, i.e.
# how important is a particular general task content category in the workforce
# Here, we're looking at non-routine cognitive analytical tasks, as defined
# by David Autor and Darron Acemoglu:

# Custom function for weighted standardization
def weighted_standardize(values, weights):
    weighted_mean = np.average(values, weights=weights)
    weighted_variance = np.average((values - weighted_mean)**2, weights=weights)
    weighted_std = np.sqrt(weighted_variance)
    return (values - weighted_mean) / weighted_std

for country in countries:
  task_cols = [f"std_{c}_{task}" for task in tasks]
  combined[f"{c}_NRCA"] = combined[task_cols].sum(axis=1)

  combined[f"std_{country}_NRCA"] = weighted_standardize(combined[f"{country}_NRCA"], combined[f"share_{country}"])

  combined[f"multip_{country}_NRCA"] = combined[f"std_{country}_NRCA"] * combined[f"share_{country}"]

agg_Spain = combined.groupby(["TIME"])["multip_Spain_NRCA"].sum().reset_index()
agg_Belgium = combined.groupby(["TIME"])["multip_Belgium_NRCA"].sum().reset_index()
agg_Poland = combined.groupby(["TIME"])["multip_Poland_NRCA"].sum().reset_index()

# Plotting the results
plt.plot(agg_Poland["TIME"], agg_Poland["multip_Poland_NRCA"])
plt.xticks(range(0, len(agg_Poland), 3), agg_Poland["TIME"][::3])
plt.show()

plt.plot(agg_Spain["TIME"], agg_Spain["multip_Spain_NRCA"])
plt.xticks(range(0, len(agg_Spain), 3), agg_Spain["TIME"][::3])
plt.show()

plt.plot(agg_Belgium["TIME"], agg_Belgium["multip_Belgium_NRCA"])
plt.xticks(range(0, len(agg_Belgium), 3), agg_Belgium["TIME"][::3])
plt.show()

# If this code gets automated and cleaned properly,
#  you should be able to easily add other countries as well as other tasks.
# E.g.:

# Routine manual
# 4.A.3.a.3	Controlling Machines and Processes
# 4.C.2.d.1.i	Spend Time Making Repetitive Motions
# 4.C.3.d.3	Pace Determined by Speed of Equipment
